# A1. NLP From Scratch: Tokens, Entities, and Topics

**Part A. What NLP Can Do**

## The Story

You just joined a media-analytics team as the ML person. On day one your manager drops a
folder of raw text on your desk: thousands of customer reviews and a pile of news articles.
No labels, no schema, no model. Two questions land on you:

1. **"Who and what do these documents mention?"** Which people, companies, and places show up?
2. **"What are they about?"** Can we surface the themes without reading every document?

This notebook is the classical toolbox that answers both - with zero model training. We will
tokenize and lemmatize text with spaCy, pull out named entities, and discover topics with
modern sentence embeddings plus clustering. Everything runs on CPU in a few minutes.

## Why start here?

The rest of this course builds toward a fine-tuned transformer running in a chatbot. Before
we get there, you need to feel the raw material: text is messy and ambiguous, and the tools
that tame it take real effort. In the very next notebook you will watch one line of
HuggingFace `pipeline()` do these same tasks - and because you saw the plumbing here, you
will understand WHY it works.

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain **why natural language is hard** for computers (ambiguity, casing, morphology).
2. Use **spaCy** to tokenize, lemmatize, and tag part-of-speech.
3. Run **Named Entity Recognition** and extract people, organizations, and places.
4. Discover **topics** in an unlabeled corpus with **sentence embeddings + KMeans**, then
   name each cluster from its top words.
5. Articulate when classical or embedding tools are enough and when to reach for transformers.

> **No deep learning training in this notebook.** We only USE pretrained tools. We start
> building models in Part B.

## Section 0. Environment Setup

We need a handful of packages. In Google Colab, run the next cell to install them. If you
are running locally in a virtualenv, you can skip the `!pip install` (but make sure these
packages are in your `requirements.txt`).

Everything in this notebook runs on **CPU** in a few minutes. A GPU is not required.

In [ ]:
# Install required packages (run this first on Google Colab).
# If running locally in a venv, you can skip this cell.
# We pin numpy<2 because several NLP libraries in this stack are not yet numpy 2 ready.
# `datasets` is HuggingFace's loader - we use it to pull the raw BBC news corpus from the Hub.

!pip install -q "spacy>=3.7,<3.9" "sentence-transformers>=2.7" "scikit-learn>=1.4" \
    "datasets>=2.19" textblob pandas matplotlib seaborn wordcloud "numpy<2"

# Download the small English spaCy pipeline. This download is SEPARATE from `pip install
# spacy` - it fetches the trained model weights we use for tokenization, POS tagging,
# lemmatization, and NER. If a later cell raises "Can't find model 'en_core_web_sm'",
# rerun this line.
!python -m spacy download en_core_web_sm

In [ ]:
# ---- Standard library ----
import re
import random
import warnings
from collections import Counter

# ---- Scientific stack ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---- Classical NLP ----
import spacy
from spacy import displacy
from textblob import TextBlob

# ---- Embeddings + clustering (modern, but still "tools" - no training by us) ----
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

# ---- Dataset loading from the HuggingFace Hub ----
from datasets import load_dataset

# ---- Word cloud for visual inspection ----
from wordcloud import WordCloud

# ---- Housekeeping ----
warnings.filterwarnings("ignore")

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---- Global knobs (edit these here, not deeper in the notebook) ----
TOUR_SIZE = 400          # how many documents we tour for speed (CPU friendly)
N_CLUSTERS = 5           # BBC news has 5 real categories; a good target for KMeans
TOP_WORDS_PER_TOPIC = 8  # how many words to print per discovered topic

print("Imports OK. sentence-transformers, spaCy, and datasets ready.")

## What Are We Building Today?

Think of this notebook as a guided tour of the classical NLP toolbox, framed by the day-one
scenario above. By the end you will have produced four concrete artifacts, all without
training a single model:

1. A **tokenized, lemmatized** view of raw text (so a computer can count words sensibly).
2. A **named-entity extractor** that lists the people and organizations in a corpus.
3. A coloured **entity visualization** you could screenshot for a stakeholder.
4. A set of **discovered topics**, each summarized by its top words.

Our tour corpus is **BBC news** - real, full-length news articles across five categories
(business, entertainment, politics, sport, tech), loaded straight from the HuggingFace Hub
(`SetFit/bbc-news`). These are the original article bodies with **natural capitalization**
("David Beckham", "Microsoft", "Tony Blair") - exactly the raw text NER and our casing
experiment need. We will pretend we do NOT know the labels (that is the realistic unlabeled
setting), then peek at them at the very end to grade ourselves. We subsample to `TOUR_SIZE`
documents so every cell runs in seconds.

In [ ]:
# Load the raw BBC news corpus from the HuggingFace Hub.
# IMPORTANT: we use the HF dataset, NOT a pre-cleaned CSV. The `text` column here is the
# ORIGINAL article body with natural capitalization and punctuation ("David Beckham",
# "Microsoft", "London") - which is exactly what Named Entity Recognition needs. A
# lowercased/stemmed copy would silently break NER, displacy, and the PERSON lab later on.
raw = load_dataset("SetFit/bbc-news", split="train")
bbc = raw.to_pandas()

# Column names in SetFit/bbc-news: 'text' (raw article) and 'label_text' (category name).
text_col = "text"        # raw, naturally-cased article body
cat_col = "label_text"   # one of: business, entertainment, politics, sport, tech

# Subsample with a fixed seed for a fast, reproducible tour.
bbc = bbc.sample(n=TOUR_SIZE, random_state=SEED).reset_index(drop=True)

print(f"BBC shape after subsample: {bbc.shape}")
print(f"Text column: '{text_col}'   Category column: '{cat_col}'")

# Quick proof that the text is RAW (capital letters and punctuation are present).
sample0 = bbc[text_col].iloc[0]
print(f"\nFirst article (first 200 chars, note the natural casing):\n{sample0[:200]}")

print("\nHidden category counts (we IGNORE these until the final self-check):")
print(bbc[cat_col].value_counts())
bbc[[cat_col, text_col]].head(3)

## Section 1. Why is NLP hard?

Computers eat numbers. Human language is messy, ambiguous, and context-dependent. A few
canonical headaches:

1. **Lexical ambiguity.** *"I saw her duck."* Did she crouch (verb), or do I see her pet
   duck (noun)? The same word, two meanings.
2. **Parsing ambiguity.** *"Time flies like an arrow; fruit flies like a banana."* Identical
   surface structure, radically different parse.
3. **Pronoun resolution.** *"The trophy doesn't fit in the suitcase because it is too big."*
   What is *it*?
4. **Negation and sarcasm.** *"This place was so great, I'll never come back."*
5. **Casing and spelling.** To a model trained on clean news text, *apple* (the fruit) and
   *Apple* (the company) are different things. Lowercase your text carelessly and entity
   recognition quietly falls apart - a trap we will hit on purpose later.

The tools in this notebook do not magically solve these. They handle them *probabilistically*:
right most of the time, wrong on edge cases. That failure mode is exactly why we INSPECT the
output instead of trusting it blindly. "Read your data first" is the oldest rule in NLP.

Here is ambiguity in action.

In [ ]:
# Load the small English pipeline once. This object is reusable: load once, use many times.
nlp = spacy.load("en_core_web_sm")

# Classic headline ambiguity: is "flies" a verb or a noun? It depends on context.
sentences = [
    "Time flies like an arrow",
    "Fruit flies like a banana",
]
for s in sentences:
    doc = nlp(s)
    tags = [(t.text, t.pos_) for t in doc]
    print(s, "->", tags)

# Notice that "flies" gets a DIFFERENT part-of-speech tag in each sentence. That is spaCy's
# statistical model resolving the ambiguity from the surrounding words - something a naive
# dictionary lookup could never do.

## Section 2. Tokenization, lemmatization, and normalization

Before any counting or clustering, we have to turn a wall of text into clean units. Three
ideas do most of the work:

- **Tokenization**: split text into words/punctuation. spaCy respects contractions and
  punctuation via its statistical pipeline; TextBlob is a simpler regex tokenizer.
- **Lemmatization**: map word variants to a dictionary base form. *running -> run*,
  *mice -> mouse*, *better -> well*. This matters for topic work: if *good* and *better* are
  two tokens, they split their weight and neither topic reads as "positive".
- **Normalization / cleanup**: lowercase, strip URLs and punctuation, collapse whitespace.
  Great for word counting and clustering - but DANGEROUS before entity recognition, because
  NER leans on capitalization. We will keep a clean version AND the raw version around.

A spaCy `Doc` exposes all of this per token:

```python
doc = nlp("Apple is looking at buying a U.K. startup for $1 billion")
for t in doc:
    print(t.text, t.lemma_, t.pos_, t.is_stop)
```

Let us see tokenization and lemmatization side by side.

In [ ]:
sample = "It's the best coffee shops I've ever visited - 10/10 would go again!"

# spaCy tokenization keeps punctuation as its own tokens; TextBlob drops it.
spacy_tokens = [t.text for t in nlp(sample)]
textblob_tokens = list(TextBlob(sample).words)
print("spaCy    :", spacy_tokens)
print("TextBlob :", textblob_tokens)

# Lemmatization: notice "shops" -> "shop" and "visited" -> "visit".
print("\nToken -> lemma:")
for t in nlp(sample):
    if t.is_alpha:
        print(f"  {t.text:10s} -> {t.lemma_}")

# A reusable, predictable cleanup helper we will lean on in the lab.
def clean_text(text: str) -> str:
    """Lowercase, drop URLs, keep only letters and spaces, collapse whitespace."""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # remove URLs
    text = re.sub(r"[^a-z\s]", " ", text)            # keep letters and whitespace only
    text = re.sub(r"\s+", " ", text).strip()         # collapse repeated whitespace
    return text

print("\nclean_text demo:")
print("  RAW  :", sample)
print("  CLEAN:", clean_text(sample))

### Lab 2.1. Build a reusable `preprocess` function

Combine the pieces above into one `preprocess(text)` function that returns a clean,
space-joined string of lemmas. Concretely it should:

1. Run `clean_text` on the input.
2. Parse the cleaned string with `nlp`.
3. Keep only tokens that are **alphabetic**, **not stopwords**, and whose **lemma** is longer
   than two characters.
4. Return those lemmas joined by single spaces.

**Why these rules?**
- Use `token.lemma_`, not `token.text`, so *shops* and *shop* collapse to one token.
- `token.is_stop` removes high-frequency filler (*the, is, and*) that drowns out topics.
- The length filter drops stray single letters left over from cleanup.

You have everything you need: `clean_text`, `nlp`, and the token attributes `is_alpha`,
`is_stop`, and `lemma_`. Build it, then run the verification cell.

**Figure: The preprocess pipeline - raw text to a clean lemma string.**

```mermaid
graph TD
    A[Raw article text] --> B[clean_text lowercase drop URLs keep letters]
    B --> C[nlp parse with spaCy]
    C --> D{Keep token?}
    D -->|alpha and not stopword and lemma len gt 2| E[Collect token lemma]
    D -->|else| F[Drop token]
    E --> G[Join lemmas into clean string]
    G --> H[bbc clean column ready for topics]
```


In [ ]:
def preprocess(text: str) -> str:
    """Clean -> parse -> keep good lemmas -> join into one string."""

    # 1. Basic regex cleanup. Reuse the helper you already have.
    #    clean_text lowercases, strips URLs and non-letters, and collapses whitespace.
    cleaned = clean_text(text)

    # 2. Parse the cleaned text with the spaCy pipeline so we get tokens with attributes
    #    (.is_alpha, .is_stop, .lemma_). nlp(...) runs the whole pipeline on the string.
    doc = nlp(cleaned)

    # 3. Build a list of lemmas. Keep a token only if it is alphabetic, is NOT a stopword,
    #    and its lemma is longer than two characters. We collect token.lemma_ (the base form),
    #    NOT token.text, so "shops" and "shop" collapse to the same token.
    #    Common mistake: using token.text here, which leaves morphological variants split.
    lemmas = [
        token.lemma_
        for token in doc
        if token.is_alpha and not token.is_stop and len(token.lemma_) > 2
    ]

    # 4. Join the surviving lemmas into a single space-separated string. Downstream TF-IDF and
    #    the embedder both expect a plain string, not a list.
    return " ".join(lemmas)


# ---- Verification (provided) ----
example = bbc[text_col].iloc[0]
print("RAW   :", example[:160])
if preprocess(example) is not None:
    out = preprocess(example)
    print("CLEAN :", out[:160])
    assert isinstance(out, str), "preprocess should return a string"
    assert " the " not in f" {out} ", "stopword 'the' should be gone"
    print("\nLooks good - preprocess returns a clean lemma string.")

In [ ]:
# Apply the pipeline to the whole tour corpus. On TOUR_SIZE=400 articles this runs in seconds.
# (For big corpora you would use nlp.pipe(texts, batch_size=64) for a 2-5x speedup; for a
#  few hundred docs the simple .apply is fine and easier to read.)
bbc["clean"] = bbc[text_col].apply(preprocess)

# Sanity check: first three cleaned articles and the average cleaned length in words.
# If Lab 2.1 was skipped, preprocess returns None for every row; we guard the display here so
# the cell does not crash, and the SAFETY-NET cell right below rebuilds a working clean column.
if bbc["clean"].iloc[0] is not None:
    for i in range(3):
        print(f"[{i}] {bbc['clean'].iloc[i][:150]}\n")
    avg_len = bbc["clean"].str.split().str.len().mean()
    print(f"Average cleaned-article length (words): {avg_len:.1f}")
else:
    print("preprocess returned None (Lab 2.1 not completed yet).")
    print("Run the SAFETY-NET cell below to build a working clean column.")

In [ ]:
# ---- SAFETY-NET (provided, do not edit) ----
# If you skipped Lab 2.1, your preprocess returns None for every row, so bbc["clean"] holds
# no usable text. Two later cells depend on it: this section's average-length check above and
# the topic-discovery demo (TfidfVectorizer.fit_transform on bbc["clean"]). Without a working
# clean column, TF-IDF would crash and the rest of the notebook would not run.
#
# This block detects the empty/None column and rebuilds it with a reference preprocess so the
# notebook keeps working. If you DID finish Lab 2.1, the probe is not None and this block is
# skipped entirely - your own preprocess is used.
clean_probe = bbc["clean"].iloc[0]
if clean_probe is None:
    print("bbc['clean'] is empty (Lab 2.1 incomplete). Building it with a reference preprocess.")

    def reference_preprocess(text):
        cleaned = clean_text(text)
        doc = nlp(cleaned)
        return " ".join(
            token.lemma_
            for token in doc
            if token.is_alpha and not token.is_stop and len(token.lemma_) > 2
        )

    bbc["clean"] = bbc[text_col].apply(reference_preprocess)
    for i in range(3):
        print(f"[{i}] {bbc['clean'].iloc[i][:150]}\n")
    avg_len = bbc["clean"].str.split().str.len().mean()
    print(f"Reference clean column ready. Average length (words): {avg_len:.1f}")
else:
    print("bbc['clean'] already built from your preprocess - safety-net not needed.")

## Section 3. Named Entity Recognition (NER)

**NER** spots spans of text that refer to real-world *things* and labels them: people,
companies, places, dates, money.

Why care? Three concrete reasons:

1. **Information extraction**: populate a search index or knowledge graph.
2. **Anonymization**: redact PII (PERSON, GPE) before sharing data.
3. **Faceted exploration**: "which organizations dominate this news feed?" - exactly the
   day-one question from our story.

spaCy's `en_core_web_sm` ships with an NER model. Calling `nlp(text)` already runs it, and
entities land at `doc.ents`. Each entity has `.text` and `.label_`.

| Label | Meaning |
|-------|---------|
| `PERSON` | People, real or fictional |
| `ORG` | Companies, agencies, institutions |
| `GPE` | Countries, cities, states |
| `LOC` | Non-GPE locations (rivers, mountain ranges) |
| `MONEY` | Monetary values |
| `DATE` | Dates and periods |

**Critical gotcha.** This model was trained on properly capitalized news text. Feed it the
lowercased `clean` column and entity recall collapses, because casing is a major signal -
*apple* is not *Apple*. **Always run NER on the original, raw text.** We prove this next.

**Figure: The casing experiment - why NER must see RAW text, not lowercased.**

```mermaid
graph TD
    A[Raw article text with natural casing] --> B[Run spaCy NER]
    B --> C[Many entities found PERSON ORG GPE]
    A --> D[Lowercase the article]
    D --> E[Run spaCy NER]
    E --> F[Few entities found]
    C --> G[Compare counts raw much greater than lowercased]
    F --> G
    G --> H[Lesson always feed RAW text to NER]
```


In [ ]:
# Run NER on the first few RAW articles. Note: use bbc[text_col] (original casing/punctuation),
# NOT bbc["clean"] - the NER model needs proper capitalization to find entities at all.
ent_rows = []
for idx, article in enumerate(bbc[text_col].head(5)):
    doc = nlp(article)
    for ent in doc.ents:
        ent_rows.append({"article_id": idx, "entity": ent.text, "label": ent.label_})

ent_df = pd.DataFrame(ent_rows)
print("Entities from the first 5 RAW articles:")
print(ent_df.head(12))

# ---- Casing experiment: prove that lowercasing breaks NER ----
# We compare entity counts on RAW vs lowercased text across several real articles. Because
# these are full naturally-cased news articles, the RAW count is much higher: capitalization
# is one of the strongest signals spaCy's NER model uses.
raw_total, lower_total = 0, 0
for article in bbc[text_col].head(10):
    raw_total += len(nlp(article).ents)
    lower_total += len(nlp(article.lower()).ents)

print(f"\nAcross 10 articles: {raw_total} entities on RAW text "
      f"vs {lower_total} on lowercased text.")
print("Lowercasing throws most entities away. This is exactly why we run NER on the RAW")
print("`text` column and never on the lowercased `clean` column.")

In [ ]:
# spaCy ships an inline visualizer. style="ent" highlights each entity with its label.
# We render the RAW article text (bbc[text_col]) so capitalization is intact and entities
# light up. SAFETY: render ONE document only. Rendering a whole corpus produces one HTML
# block per document and will freeze the notebook.
doc = nlp(bbc[text_col].iloc[0])
displacy.render(doc, style="ent", jupyter=True)

### Lab 3.1. Who shows up most? Top PERSON entities

Your turn. Using the fast batched API `nlp.pipe`, find the people mentioned most often in the
tour corpus.

Steps:

1. Iterate over `bbc[text_col]` using `nlp.pipe(...)`. Pass `batch_size=32` and
   `disable=["parser", "tagger", "lemmatizer"]` so spaCy only runs the NER component (much
   faster - calling `nlp()` in a plain loop is the classic beginner mistake).
2. For every entity in each doc, keep only those whose label is `"PERSON"`.
3. Count occurrences in a `Counter`, storing `ent.text.strip()` (the `.strip()` keeps
   "Tony Blair" and "Tony Blair " from counting as two different people).
4. Print the **top 10** with `.most_common(10)`.

**Stretch (entity normalization).** Real names appear in variants: *Tony Blair*, *Blair*,
*Mr Blair*. Counting them separately understates the true frequency. Write a tiny normalizer
that maps each PERSON entity to its **last token** (a crude surname key), recount, and compare
the top 10 to your raw count. This is a baby version of *entity linking* / *grounding* - the
real production task of mapping surface mentions to canonical identities.

**Homework extension (co-occurrence).** Build an ORG-and-GPE co-occurrence view: for each
article, collect the set of `ORG` and `GPE` entities, then count which (ORG, GPE) pairs appear
together most often. This is the seed of a knowledge graph. Bonus: think about how news-trained
NER would degrade on noisy product reviews or tweets (domain shift) and how you would detect it.

In [ ]:
person_counter = Counter()

# 1. Loop over the corpus with nlp.pipe for speed. nlp.pipe streams the texts through the
#    pipeline in batches, which is much faster than calling nlp() once per document in a plain
#    loop. We disable parser/tagger/lemmatizer because we only need the NER component here -
#    running the others would just waste time.
for doc in nlp.pipe(bbc[text_col], batch_size=32,
                    disable=["parser", "tagger", "lemmatizer"]):
    # 2. Look at every entity in this doc.
    for ent in doc.ents:
        # 3. Keep only PERSON entities, then count the stripped text.
        #    .strip() collapses "Tony Blair" and "Tony Blair " into one key.
        if ent.label_ == "PERSON":
            person_counter[ent.text.strip()] += 1

# 4. Top 10 most mentioned people. Counter.most_common(10) returns (name, count) pairs.
top_people = person_counter.most_common(10)
print(top_people)

# ---- Verification (provided) ----
if top_people is not None:
    assert len(top_people) <= 10, "should be at most the top 10"
    assert all(isinstance(name, str) for name, _ in top_people), "keys should be names"
    print("\nNice - you extracted the most-mentioned people with batched NER.")

## Section 4. Discovering topics with sentence embeddings

We never trained a topic model the old way (LDA, which treats text as a bag of independent
words and ignores word order). Instead we use the modern, simpler recipe that also previews
the most important idea in this course: **turn each document into a vector, then cluster the
vectors.**

The recipe:

1. **Embed.** A pretrained `sentence-transformers` model maps each article to a fixed-size
   vector (384 numbers) that captures meaning. We use `all-MiniLM-L6-v2` - a distilled,
   6-layer BERT, only ~22 MB, fast on CPU. Documents about the same theme land near each
   other in this space.
2. **Cluster.** `KMeans` groups the vectors into `k` clusters. Each cluster is a discovered
   topic. We set `n_init=10` and `random_state=42` so the result is reproducible (KMeans
   starts from random centroids; without these it can drift run to run).
3. **Name.** A cluster of vectors is not human-readable, so we run a per-cluster TF-IDF over
   the cleaned text and read off each cluster's most distinctive words. Those top words ARE
   the topic label.

> This `all-MiniLM-L6-v2` embedder is the through-line of the course. In Part B you use the
> same "text -> vector" idea as features for a small neural net; in Part C the model that
> produces these vectors becomes the thing you fine-tune. Meet it now.

Here it is end to end.

**Figure: Topic discovery - embed, cluster with KMeans, then name each cluster.**

```mermaid
graph TD
    A[Raw articles] --> B[SentenceTransformer all-MiniLM-L6-v2]
    B --> C[384-dim vector per document]
    C --> D[KMeans n_clusters 5 fixed seed]
    D --> E[Cluster id per document]
    E --> F[Per-cluster TF-IDF on cleaned text]
    F --> G[Top words per cluster]
    G --> H[Each cluster named as a topic]
```


In [ ]:
# 1. EMBED: load the sentence-transformer and encode the RAW articles (bbc[text_col]).
#    We pass raw text, NOT the cleaned column, because this model was trained on natural
#    sentences: it uses word order, casing, stopwords, and punctuation to build meaning.
#    Stripping all that (the `clean` column) would throw away signal the embedder relies on.
#    ~22 MB download, runs on CPU.
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(bbc[text_col].tolist(), show_progress_bar=False)
print(f"Embeddings shape: {embeddings.shape}  (documents x 384-dim vectors)")

# 2. CLUSTER: KMeans on the embeddings. n_init=10 + fixed seed = reproducible clusters.
kmeans = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=SEED)
bbc["cluster"] = kmeans.fit_predict(embeddings)
print("Documents per cluster:")
print(bbc["cluster"].value_counts().sort_index())

# 3. NAME: run TF-IDF over the CLEANED text, then for each cluster read off the words with
#    the highest average TF-IDF weight inside that cluster. Those words name the topic.
tfidf = TfidfVectorizer(min_df=3, max_df=0.5)
tfidf_matrix = tfidf.fit_transform(bbc["clean"])
terms = np.array(tfidf.get_feature_names_out())

print("\nDiscovered topics (top words per cluster):")
for c in range(N_CLUSTERS):
    rows = (bbc["cluster"] == c).values
    mean_weights = np.asarray(tfidf_matrix[rows].mean(axis=0)).ravel()
    top_idx = mean_weights.argsort()[::-1][:TOP_WORDS_PER_TOPIC]
    print(f"  Cluster {c}: {', '.join(terms[top_idx])}")

### Lab 4.1. Did the clusters rediscover the real categories?

This is the main self-check of the notebook. BBC has 5 hidden categories (business,
entertainment, politics, sport, tech) and we asked KMeans for 5 clusters. Did clustering on
sentence embeddings rediscover the human labels - without ever seeing them?

Steps:

1. Build a **cross-tabulation** of cluster id (`bbc["cluster"]`) vs the true category
   (`bbc[cat_col]`) with `pd.crosstab`.
2. Draw it as a heatmap with `sns.heatmap(..., annot=True, fmt="d", cmap="Blues")`.
3. In one sentence, say which clusters cleanly map to which categories. A clean recovery has
   one dominant cell per row.

Expect strong recovery: sentence embeddings usually separate sport, tech, and entertainment
cleanly; business and politics sometimes bleed together.

**Stretch (choosing k).** We assumed k=5 because we secretly knew the answer. In the real
unlabeled setting you must choose k. Loop k over `range(2, 9)`, fit KMeans for each, compute
`silhouette_score(embeddings, labels)` from `sklearn.metrics`, and plot score vs k. Does the
best silhouette agree with the true 5? Combine with an elbow (inertia) plot for a second
opinion.

**Homework extension (compare to BERTopic).** Install `bertopic`, fit it on the same raw
articles, and compare its discovered topics to your KMeans-plus-TF-IDF topics. BERTopic adds
UMAP dimensionality reduction and HDBSCAN (which picks the number of topics for you and flags
outliers as cluster -1). Which gives more coherent, more specific topics on this corpus? When
would the extra machinery be worth it in production?

In [ ]:
# 1. Cross-tabulate discovered cluster vs true category.
#    pd.crosstab(rows, cols) counts how many documents fall into each (cluster, category)
#    pair. Rows = the KMeans cluster id, columns = the hidden true label. A clean recovery
#    shows one big number per row (each cluster maps mostly to one category).
crosstab = pd.crosstab(bbc["cluster"], bbc[cat_col])

print(crosstab)

# 2. Heatmap of the crosstab.
fig, ax = plt.subplots(figsize=(8, 5))
# annot=True prints the counts inside each cell; fmt="d" formats them as integers.
sns.heatmap(crosstab, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("KMeans cluster vs true BBC category (one big cell per row = clean recovery)")
ax.set_xlabel("True category")
ax.set_ylabel("Discovered cluster")
plt.tight_layout()
plt.show()

# ---- Verification (provided) ----
if crosstab is not None:
    assert crosstab.values.sum() == len(bbc), "crosstab should cover every document"
    print("\nGreat - you compared unsupervised clusters against the hidden labels.")

## Wrap-up

### What you built (zero training)

- A reusable **clean -> tokenize -> lemmatize** pipeline (`clean_text`, `preprocess`).
- A **named-entity extractor** that surfaces the people and organizations in a corpus, plus
  an inline visualization - and a hard-won lesson that casing matters for NER.
- A **topic discovery** pipeline using sentence embeddings + KMeans, with each topic named by
  its top TF-IDF words, that rediscovered the hidden BBC categories.

### What you did NOT do here (on purpose)

- No model **training** - we only USED pretrained tools. Training starts in Part B.
- No deep dive into LDA or BERTopic internals - we picked the simplest recipe that works.
- No transformers yet - but you just met the sentence embedder that becomes their foundation.

### When is this toolbox enough?

| Situation | Reach for |
|-----------|-----------|
| Extract names/places, redact PII | spaCy NER (rules + small model, fast, cheap) |
| Group unlabeled docs by theme | sentence embeddings + KMeans |
| Quick, interpretable baseline before any DL | clean + count + cluster, every time |
| Nuanced meaning, sarcasm, negation, real accuracy | transformers (next notebooks) |

Start simple. A clean baseline is fast, interpretable, and surprisingly hard to beat - so it
is always worth building before you reach for a heavier model.

### Bridge to A2

All of this took installs, custom stopwords, casing care, and clustering plumbing. In the
**next notebook (A2: Pipeline Tour)** you will solve these same tasks - sentiment, NER,
zero-shot classification, question answering - in essentially **one line each** with
HuggingFace `pipeline()`. The payoff: because you saw the raw material here, you will
understand exactly what that one line is doing under the hood.

Note on the running example: here we explored a BBC news corpus to learn the tools. From A2 onward the course follows one scenario - a customer-support platform - so the same skills (NER, embeddings, zero-shot) now serve support tickets instead of news articles. Same toolbox, one product story all the way to the chatbot.

**Figure: The bridge from the A1 classical toolbox to the A2 one-line pipeline.**

```mermaid
flowchart LR
    A[A1 classical toolbox] --> B[Install packages and download model]
    B --> C[Custom clean and stopword care]
    C --> D[spaCy NER and embeddings plus KMeans]
    D --> E[Entities and topics with effort]
    E --> F[A2 HuggingFace pipeline]
    F --> G[One line per task]
    G --> H[Same outputs much less plumbing]
```
